# 🐾 Desafio CP06 — Construindo um Compilador Completo
## CasaScript 2.0 — Automação Residencial para Pets (Pet Tech)

**Integrantes:**

*   Diego Azevedo Dias Ignacio - 2392072
*   Gabriella Pereira Rodrigues - 2334425
*   Giovanna Bispo Da Silva - 2367880
*   José Henrique Guimarães Galvão Silva - 2499948

**Exemplo de entrada do domínio:**

```casascript
QUANDO relogio >= 08:00 E relogio <= 09:00 ENTAO dispensar(comedouro, 100); notificar("Hora do café!") SENAO notificar("Fora do horário") FIM
```

Este notebook implementa um **compilador completo de ponta a ponta** com [Lark](https://lark-parser.readthedocs.io/) para a linguagem **CasaScript 2.0** focada no domínio de Pet Tech, com:
análise léxica (literal de `HORARIO` HH:MM em minutos), análise sintática (suporte a blocos `SENAO`), análise semântica com Tabela de Símbolos do cenário Pet e avisos de inconsistência (ações/condições duplicadas), otimizador de AST (eliminação de dupla negação `NAO NAO`, identidades aritméticas, *constant folding* e código morto), gerador de bytecode para máquina de pilha, Máquina Virtual (VM) com runtime orientado a eventos (disparos em borda de subida e descida) e bateria de testes automáticos (RF6).

## 1) Preparação do Ambiente

In [1]:
# CÉLULA 1: Instalação de dependências e importações
%pip install -q lark ipywidgets

import html
import time
import difflib
import operator
from dataclasses import dataclass, field
import lark
from lark import Lark, Transformer, v_args
from lark.exceptions import UnexpectedCharacters, UnexpectedToken, UnexpectedEOF
import ipywidgets as widgets
from IPython.display import display, HTML

print("Ambiente pronto! Lark versão:", lark.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 23.9 MB/s eta 0:00:00
Ambiente pronto! Lark versão: 1.3.1


## 2) Ferramentas de Visualização

In [4]:
# CÉLULA 2: Funções auxiliares de exibição HTML
def tabela(cabecalho, linhas, titulo=None):
    """Exibe dados em formato de tabela HTML estilizada."""
    th = "".join(f"<th style='padding: 4px 12px; border-bottom:2px solid #888; text-align:left'>{c}</th>" for c in cabecalho)
    corpo = ""
    for linha in linhas:
        tds = "".join(f"<td style='padding:3px 12px; border-bottom:1px solid #ccc; font-family:monospace'>{html.escape(str(c))}</td>" for c in linha)
        corpo += f"<tr>{tds}</tr>"
    topo = f"<b>{titulo}</b><br>" if titulo else ""
    display(HTML(f"{topo}<table style='border-collapse:collapse; margin:6px 0;'><tr>{th}</tr>{corpo}</table>"))

def caixa(texto, tipo="info"):
    """Exibe caixa de mensagem colorida ('ok', 'erro', 'aviso', 'info')."""
    cores = {"ok": "#1b7f3b", "erro": "#b3261e", "aviso": "#b36b00", "info": "#1f5fa8"}
    display(HTML(f"<div style='border-left:5px solid {cores[tipo]}; padding:6px 12px; margin:6px 0; font-family:monospace; white-space: pre-wrap'>{html.escape(texto)}</div>"))

print("Ferramentas de visualização carregadas!")

Ferramentas de visualização carregadas!


## 3) Gramática CasaScript 2.0 (RF1 e RF2)

In [26]:
# CÉLULA 3: Gramática LALR(1) com suporte a HORARIO e SENAO (Com Prioridade Léxica)
GRAMATICA_PET = r"""
programa : regra+

regra    : "QUANDO" expr "ENTAO" acoes ["SENAO" acoes] "FIM"

acoes    : acao (";" acao)* ";"?

acao     : NOME "(" [argumentos] ")"
argumentos: expr ("," expr)*

?expr       : ou
?ou         : e | ou "OU" e           -> op_ou
?e          : nao | e "E" nao         -> op_e
?nao        : comparacao | "NAO" nao  -> op_nao
?comparacao : soma | soma OP_COMP soma -> op_comp
?soma       : produto | soma OP_SOMA produto -> op_arit
?produto    : atomo | produto OP_MULT atomo   -> op_arit

?atomo      : NUMERO                  -> numero
            | HORARIO                 -> horario
            | TEXTO                   -> texto
            | "VERDADEIRO"            -> verdadeiro
            | "FALSO"                 -> falso
            | NOME                    -> variavel
            | "(" expr ")"

// TERMINAIS LÉXICOS
OP_COMP  : ">=" | "<=" | "!=" | "==" | ">" | "<"
OP_SOMA  : "+" | "-"
OP_MULT  : "*" | "/"

// HORARIO.2 força prioridade sobre NUMERO para capturar 08:00 antes de tentar ler 08 como número
HORARIO.2: /(?:[01]\d|2[0-3]):[0-5]\d/
NUMERO   : /\d+(\.\d+)?/
TEXTO    : /"[^"\r\n]*"/
NOME     : /[a-z_][a-z0-9_]*/
COMENTARIO: /#[^\r\n]*/

%import common.WS
%ignore WS
%ignore COMENTARIO
"""

parser_pet = Lark(GRAMATICA_PET, start="programa", parser="lalr", propagate_positions=True)
print("Gramática CasaScript 2.0 (Pets) carregada com sucesso!")

Gramática CasaScript 2.0 (Pets) carregada com sucesso!


## 4) Construção da AST (RF1 e RF2)

In [16]:
# CÉLULA 4: Estrutura dos nós da AST e Transformer
@dataclass
class Numero:
    valor: float; linha: int = 0; coluna: int = 0

@dataclass
class Horario:
    valor: int; linha: int = 0; coluna: int = 0  # minutos desde meia-noite

@dataclass
class Texto:
    valor: str; linha: int = 0; coluna: int = 0

@dataclass
class Logico:
    valor: bool; linha: int = 0; coluna: int = 0

@dataclass
class Variavel:
    nome: str; linha: int = 0; coluna: int = 0

@dataclass
class Aritmetica:
    op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0

@dataclass
class Comparacao:
    op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0

@dataclass
class Logica:
    op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0

@dataclass
class Nao:
    expr: object; linha: int = 0; coluna: int = 0

@dataclass
class Acao:
    nome: str; args: list; linha: int = 0; coluna: int = 0

@dataclass
class Regra:
    numero: int; condicao: object; acoes_entao: list; acoes_senao: list; fonte: str = ""; linha: int = 0; coluna: int = 0

@dataclass
class Programa:
    regras: list

@v_args(meta=True)
class ConstrutorASTPet(Transformer):
    def __init__(self, fonte):
        super().__init__()
        self.fonte = fonte

    def numero(self, meta, f):
        t = f[0]
        return Numero(float(t) if "." in t else int(t), meta.line, meta.column)

    def horario(self, meta, f):
        hh, mm = map(int, str(f[0]).split(":"))
        return Horario(hh * 60 + mm, meta.line, meta.column)

    def texto(self, meta, f):
        return Texto(f[0][1:-1], meta.line, meta.column)

    def verdadeiro(self, meta, f): return Logico(True, meta.line, meta.column)
    def falso(self, meta, f): return Logico(False, meta.line, meta.column)
    def variavel(self, meta, f): return Variavel(str(f[0]), meta.line, meta.column)

    def op_arit(self, meta, f): return Aritmetica(str(f[1]), f[0], f[2], meta.line, meta.column)
    def op_comp(self, meta, f): return Comparacao(str(f[1]), f[0], f[2], meta.line, meta.column)
    def op_e(self, meta, f): return Logica("E", f[0], f[1], meta.line, meta.column)
    def op_ou(self, meta, f): return Logica("OU", f[0], f[1], meta.line, meta.column)
    def op_nao(self, meta, f): return Nao(f[0], meta.line, meta.column)

    def argumentos(self, meta, f): return list(f)
    def acao(self, meta, f): return Acao(str(f[0]), f[1] or [], meta.line, meta.column)
    def acoes(self, meta, f): return list(f)

    def regra(self, meta, f):
        cond = f[0]
        entao = f[1]
        senao = f[2] if len(f) > 2 and f[2] is not None else []
        trecho = self.fonte[meta.start_pos:meta.end_pos]
        return Regra(0, cond, entao, senao, trecho, meta.line, meta.column)

    def programa(self, meta, f):
        for i, r in enumerate(f, 1): r.numero = i
        return Programa(list(f))

def construir_ast_pet(fonte):
    return ConstrutorASTPet(fonte).transform(parser_pet.parse(fonte))

print("Construtor da AST pronto!")

Construtor da AST pronto!


## 5) Tabela de Símbolos e Análise Semântica (RF3)

In [17]:
# CÉLULA 5: Tabela de Símbolos do cenário Casa para Pets e Analisador Semântico
SENSORES_PET = {
    "peso_racao": "numero",
    "nivel_agua": "numero",
    "temperatura": "numero",
    "presenca_pet": "logico",
    "status_pet": "texto",
    "relogio": "horario"
}

FAIXA_SENSOR_PET = {
    "peso_racao": (0, 2000),
    "nivel_agua": (0, 100),
    "temperatura": (-10, 50),
    "relogio": (0, 1439)
}

VALORES_TEXTO_PET = {
    "status_pet": {"dormindo", "brincando", "comendo", "ausente"}
}

DISPOSITIVOS_PET = {
    "comedouro": "comedouro",
    "fonte_agua": "fonte",
    "climatizador": "climatizador",
    "camera": "camera",
    "tigela_agua": "tigela"
}

ACOES_PET = {
    "dispensar": (["dispositivo", "numero"], {"comedouro"}),
    "trocar_agua": (["dispositivo"], {"fonte", "tigela"}),
    "ligar": (["dispositivo"], {"fonte", "climatizador", "camera"}),
    "desligar": (["dispositivo"], {"fonte", "climatizador", "camera"}),
    "ajustar": (["dispositivo", "numero"], {"climatizador"}),
    "notificar": (["texto"], set())
}

FAIXA_AJUSTE_PET = {
    "comedouro": (10, 500),
    "climatizador": (16, 30)
}

def repr_no(no):
    if isinstance(no, Numero): return str(no.valor)
    if isinstance(no, Horario): return f"{no.valor//60:02d}:{no.valor%60:02d}"
    if isinstance(no, Texto): return f'"{no.valor}"'
    if isinstance(no, Logico): return "VERDADEIRO" if no.valor else "FALSO"
    if isinstance(no, Variavel): return no.nome
    if isinstance(no, Nao): return f"NAO {repr_no(no.expr)}"
    if isinstance(no, (Aritmetica, Comparacao, Logica)): return f"({repr_no(no.esq)} {no.op} {repr_no(no.dir)})"
    if isinstance(no, Acao): return f"{no.nome}({', '.join(repr_no(a) for a in no.args)})"
    return str(no)

class AnalisadorSemanticoPet:
    def __init__(self):
        self.erros = []
        self.avisos = []

    def erro(self, no, msg): self.erros.append(f"Linha {no.linha}, col {no.coluna}: {msg}")
    def aviso(self, no, msg): self.avisos.append(f"Linha {no.linha}, col {no.coluna}: {msg}")

    def sugestao(self, nome, conhecidos):
        parecido = difflib.get_close_matches(nome, conhecidos, n=1)
        return f" Você quis dizer '{parecido[0]}'?" if parecido else ""

    def tipo(self, no):
        if isinstance(no, Numero): return "numero"
        if isinstance(no, Horario): return "horario"
        if isinstance(no, Texto): return "texto"
        if isinstance(no, Logico): return "logico"
        if isinstance(no, Variavel):
            if no.nome in SENSORES_PET: return SENSORES_PET[no.nome]
            if no.nome in DISPOSITIVOS_PET:
                self.erro(no, f"'{no.nome}' é um DISPOSITIVO, não um sensor.")
                return "erro"
            self.erro(no, f"sensor '{no.nome}' não existe." + self.sugestao(no.nome, SENSORES_PET))
            return "erro"
        if isinstance(no, Aritmetica):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != "numero" or td != "numero":
                self.erro(no, f"operação '{no.op}' só aceita números.")
                return "erro"
            return "numero"
        if isinstance(no, Comparacao):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != td:
                self.erro(no, f"não dá para comparar {te} com {td} usando '{no.op}'.")
                return "erro"
            return "logico"
        if isinstance(no, Logica):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != "logico" or td != "logico":
                self.erro(no, f"'{no.op}' só opera com condições lógicas.")
                return "erro"
            return "logico"
        if isinstance(no, Nao):
            t = self.tipo(no.expr)
            if t not in ("logico", "erro"):
                self.erro(no, f"NAO só pode negar expressões lógicas.")
                return "erro"
            return t
        return "erro"

    def verificar_acao(self, acao):
        if acao.nome not in ACOES_PET:
            self.erro(acao, f"ação '{acao.nome}' não existe." + self.sugestao(acao.nome, ACOES_PET))
            return
        params, aceita = ACOES_PET[acao.nome]
        if len(acao.args) != len(params):
            self.erro(acao, f"'{acao.nome}' espera {len(params)} arg(s), mas recebeu {len(acao.args)}.")
            return
        for param, arg in zip(params, acao.args):
            if param == "dispositivo":
                if not isinstance(arg, Variavel) or arg.nome not in DISPOSITIVOS_PET:
                    nome = arg.nome if isinstance(arg, Variavel) else "?"
                    self.erro(arg, f"'{acao.nome}' exige dispositivo válido ('{nome}' incorreto).")
                    return
                tipo_disp = DISPOSITIVOS_PET[arg.nome]
                if tipo_disp not in aceita:
                    self.erro(acao, f"dispositivo '{arg.nome}' não aceita a ação '{acao.nome}'.")
                    return
            else:
                t = self.tipo(arg)
                if t != "erro" and t != param:
                    self.erro(arg, f"'{acao.nome}' esperava argumento do tipo {param}, mas recebeu {t}.")

        if params == ["dispositivo", "numero"] and isinstance(acao.args[1], Numero):
            disp_nome = acao.args[0].nome
            if disp_nome in DISPOSITIVOS_PET:
                cat = DISPOSITIVOS_PET[disp_nome]
                if cat in FAIXA_AJUSTE_PET:
                    f_min, f_max = FAIXA_AJUSTE_PET[cat]
                    if not (f_min <= acao.args[1].valor <= f_max):
                        self.erro(acao.args[1], f"valor {acao.args[1].valor} fora da faixa ({f_min} a {f_max}) para '{disp_nome}'.")

    def analisar(self, programa):
        condicoes_vistas = {}
        for regra in programa.regras:
            t = self.tipo(regra.condicao)
            if t not in ("logico", "erro"):
                self.erro(regra.condicao, f"condição da Regra #{regra.numero} precisa ser lógica.")

            cond_str = repr_no(regra.condicao)
            if cond_str in condicoes_vistas:
                self.aviso(regra, f"Regra #{regra.numero} possui condição idêntica à Regra #{condicoes_vistas[cond_str]}.")
            else:
                condicoes_vistas[cond_str] = regra.numero

            for bloco, acoes in [("ENTAO", regra.acoes_entao), ("SENAO", regra.acoes_senao)]:
                acoes_vistas = set()
                for acao in acoes:
                    self.verificar_acao(acao)
                    str_acao = repr_no(acao)
                    if str_acao in acoes_vistas:
                        self.aviso(acao, f"Ação duplicada '{str_acao}' no bloco {bloco} da Regra #{regra.numero}.")
                    acoes_vistas.add(str_acao)

        return self.erros, self.avisos

print("Analisador semântico para Pets pronto!")

Analisador semântico para Pets pronto!


## 6) Otimizador com Novas Técnicas (RF4)

In [18]:
# CÉLULA 6: Otimizador de AST (RF4)
OPS_ARIT = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}
OPS_COMP = {"==": operator.eq, "!=": operator.ne, ">": operator.gt, "<": operator.lt, ">=": operator.ge, "<=": operator.le}

class OtimizadorPet:
    def __init__(self):
        self.relatorio = []

    def registrar(self, tecnica, antes, depois):
        self.relatorio.append((tecnica, repr_no(antes), repr_no(depois)))

    def otimizar_expr(self, no):
        if isinstance(no, (Aritmetica, Comparacao, Logica)):
            no = type(no)(no.op, self.otimizar_expr(no.esq), self.otimizar_expr(no.dir), no.linha, no.coluna)
        elif isinstance(no, Nao):
            no = Nao(self.otimizar_expr(no.expr), no.linha, no.coluna)

        if isinstance(no, Nao) and isinstance(no.expr, Nao):
            novo = no.expr.expr
            self.registrar("Eliminação de dupla negação", no, novo)
            return self.otimizar_expr(novo)

        if isinstance(no, Aritmetica):
            if no.op == "*":
                if (isinstance(no.esq, Numero) and no.esq.valor == 0) or (isinstance(no.dir, Numero) and no.dir.valor == 0):
                    novo = Numero(0, no.linha, no.coluna)
                    self.registrar("Identidade aritmética (x * 0 = 0)", no, novo)
                    return novo

            if isinstance(no.dir, Numero):
                if no.op in ("+", "-") and no.dir.valor == 0:
                    self.registrar(f"Identidade aritmética (x {no.op} 0)", no, no.esq)
                    return no.esq
                if no.op in ("*", "/") and no.dir.valor == 1:
                    self.registrar(f"Identidade aritmética (x {no.op} 1)", no, no.esq)
                    return no.esq

            if isinstance(no.esq, Numero):
                if no.op == "+" and no.esq.valor == 0:
                    self.registrar("Identidade aritmética (0 + x)", no, no.dir)
                    return no.dir
                if no.op == "*" and no.esq.valor == 1:
                    self.registrar("Identidade aritmética (1 * x)", no, no.dir)
                    return no.dir

        if isinstance(no, Aritmetica) and isinstance(no.esq, Numero) and isinstance(no.dir, Numero):
            res = OPS_ARIT[no.op](no.esq.valor, no.dir.valor)
            novo = Numero(int(res) if isinstance(res, float) and res.is_integer() else res, no.linha, no.coluna)
            self.registrar("Constant folding", no, novo)
            return novo

        if isinstance(no, Comparacao) and type(no.esq) == type(no.dir) and isinstance(no.esq, (Numero, Horario, Texto, Logico)):
            res = OPS_COMP[no.op](no.esq.valor, no.dir.valor)
            novo = Logico(res, no.linha, no.coluna)
            self.registrar("Constant folding", no, novo)
            return novo

        if isinstance(no, Logica):
            for const, outro in ((no.esq, no.dir), (no.dir, no.esq)):
                if isinstance(const, Logico):
                    if no.op == "E":
                        novo = outro if const.valor else Logico(False, no.linha, no.coluna)
                    else:
                        novo = Logico(True, no.linha, no.coluna) if const.valor else outro
                    self.registrar("Simplificação lógica", no, novo)
                    return self.otimizar_expr(novo)

        return no

    def otimizar(self, programa):
        regras = []
        for r in programa.regras:
            cond = self.otimizar_expr(r.condicao)
            if isinstance(cond, Logico) and not cond.valor:
                self.relatorio.append(("Código morto", f"Regra #{r.numero}", "removida (condição sempre FALSO)"))
                continue
            regras.append(Regra(r.numero, cond, r.acoes_entao, r.acoes_senao, r.fonte, r.linha, r.coluna))
        return Programa(regras)

print("Otimizador RF4 pronto!")

Otimizador RF4 pronto!


## 7) Geração de Código Bytecode (RF5)

In [19]:
# CÉLULA 7: Gerador de Bytecode (RF5)
@dataclass
class RegraCompiladaPet:
    numero: int
    fonte: str
    condicao: list
    acoes_entao: list
    acoes_senao: list

class GeradorDeCodigoPet:
    def __init__(self):
        self.codigo = []

    def emitir(self, op, arg=None):
        self.codigo.append([op, arg])
        return len(self.codigo) - 1

    def gerar_expr(self, no):
        if isinstance(no, (Numero, Texto, Logico, Horario)):
            self.emitir("PUSH_CONST", no.valor)
        elif isinstance(no, Variavel):
            self.emitir("LOAD_SENSOR", no.nome)
        elif isinstance(no, Aritmetica):
            self.gerar_expr(no.esq); self.gerar_expr(no.dir)
            self.emitir("BIN_OP", no.op)
        elif isinstance(no, Comparacao):
            self.gerar_expr(no.esq); self.gerar_expr(no.dir)
            self.emitir("COMPARE", no.op)
        elif isinstance(no, Nao):
            self.gerar_expr(no.expr)
            self.emitir("NOT")
        elif isinstance(no, Logica):
            self.gerar_expr(no.esq)
            salto = self.emitir("JUMP_IF_FALSE_OR_POP" if no.op == "E" else "JUMP_IF_TRUE_OR_POP", "?")
            self.gerar_expr(no.dir)
            self.codigo[salto][1] = len(self.codigo)

    def gerar_condicao(self, expr):
        self.codigo = []
        self.gerar_expr(expr)
        self.emitir("RETURN")
        return self.codigo

    def gerar_bloco_acoes(self, acoes):
        self.codigo = []
        for acao in acoes:
            params = ACOES_PET[acao.nome][0]
            for param, arg in zip(params, acao.args):
                if param == "dispositivo":
                    self.emitir("LOAD_DEVICE", arg.nome)
                else:
                    self.gerar_expr(arg)
            self.emitir("CALL_ACTION", (acao.nome, len(acao.args)))
        self.emitir("HALT")
        return self.codigo

    def gerar(self, programa):
        res = []
        for r in programa.regras:
            cond = self.gerar_condicao(r.condicao)
            entao = self.gerar_bloco_acoes(r.acoes_entao)
            senao = self.gerar_bloco_acoes(r.acoes_senao)
            res.append(RegraCompiladaPet(r.numero, r.fonte, cond, entao, senao))
        return res

print("Gerador de Bytecode RF5 pronto!")

Gerador de Bytecode RF5 pronto!


## 8) Máquina Virtual e Ambiente CasaPet (RF5)

In [20]:
# CÉLULA 8: Classe CasaPet e Máquina Virtual (RF5)
class CasaPet:
    def __init__(self):
        self.sensores = {
            "peso_racao": 500,
            "nivel_agua": 80,
            "temperatura": 22,
            "presenca_pet": False,
            "status_pet": "dormindo",
            "relogio": 480
        }
        self.dispositivos = {
            "comedouro": {"porcoes": 0},
            "fonte_agua": {"ligado": True},
            "climatizador": {"ligado": False, "nivel": 22},
            "camera": {"ligado": False},
            "tigela_agua": {"limpa": True}
        }
        self.log = []

    def executar_acao(self, nome, args):
        hh, mm = self.sensores["relogio"] // 60, self.sensores["relogio"] % 60
        tag = f"[{hh:02d}:{mm:02d}]"

        if nome == "dispensar":
            self.dispositivos[args[0]]["porcoes"] += args[1]
            self.log.append(f"{tag} DISPENSAR: {args[1]}g de ração em '{args[0]}'")
        elif nome == "trocar_agua":
            self.log.append(f"{tag} TROCAR ÁGUA: Drenando e reabastecendo '{args[0]}'")
        elif nome in ("ligar", "desligar"):
            self.dispositivos[args[0]]["ligado"] = (nome == "ligar")
            self.log.append(f"{tag} {nome.upper()}: '{args[0]}'")
        elif nome == "ajustar":
            self.dispositivos[args[0]].update(ligado=True, nivel=args[1])
            self.log.append(f"{tag} AJUSTAR: '{args[0]}' para {args[1]}°C")
        elif nome == "notificar":
            self.log.append(f"{tag} NOTIFICAÇÃO: {args[0]}")

class MaquinaVirtualPet:
    def __init__(self, casa):
        self.casa = casa

    def executar(self, codigo):
        pilha, pc = [], 0
        while True:
            op, arg = codigo[pc]
            pc += 1
            if op == "PUSH_CONST": pilha.append(arg)
            elif op == "LOAD_SENSOR": pilha.append(self.casa.sensores[arg])
            elif op == "LOAD_DEVICE": pilha.append(arg)
            elif op == "BIN_OP":
                b, a = pilha.pop(), pilha.pop()
                pilha.append(OPS_ARIT[arg](a, b))
            elif op == "COMPARE":
                b, a = pilha.pop(), pilha.pop()
                pilha.append(OPS_COMP[arg](a, b))
            elif op == "NOT": pilha.append(not pilha.pop())
            elif op == "JUMP_IF_FALSE_OR_POP":
                if not pilha[-1]: pc = arg
                else: pilha.pop()
            elif op == "JUMP_IF_TRUE_OR_POP":
                if pilha[-1]: pc = arg
                else: pilha.pop()
            elif op == "CALL_ACTION":
                nome, narg = arg
                args = pilha[-narg:]
                del pilha[-narg:]
                self.casa.executar_acao(nome, args)
            elif op == "RETURN": return pilha.pop()
            elif op == "HALT": return None

print("Máquina Virtual e Ambiente CasaPet prontos!")

Máquina Virtual e Ambiente CasaPet prontos!


## 9) Central de Automação com Borda de Descida (RF5)

In [21]:
# CÉLULA 9: Central de Automação com avaliação de borda (RF5)
class CentralAutomaçaoPet:
    def __init__(self, bytecode, casa):
        self.regras = bytecode
        self.casa = casa
        self.vm = MaquinaVirtualPet(casa)
        self.estado_anterior = {r.numero: False for r in bytecode}

    def avaliar_todas(self):
        disparadas = []
        for regra in self.regras:
            agora = bool(self.vm.executar(regra.condicao))
            anterior = self.estado_anterior[regra.numero]

            if agora and not anterior:
                self.vm.executar(regra.acoes_entao)
                disparadas.append(f"#{regra.numero} (ENTAO)")
            elif not agora and anterior:
                if regra.acoes_senao:
                    self.vm.executar(regra.acoes_senao)
                    disparadas.append(f"#{regra.numero} (SENAO)")

            self.estado_anterior[regra.numero] = agora
        return disparadas

    def evento(self, sensor, valor):
        self.casa.sensores[sensor] = valor
        return self.avaliar_todas()

print("Central de Automação com borda de descida pronta!")

Central de Automação com borda de descida pronta!


## 10) Compilador de Ponta a Ponta

In [22]:
# CÉLULA 10: Pipeline do Compilador Completo
NOMES_AMIGAVEIS = {
    "QUANDO": "QUANDO", "ENTAO": "ENTAO", "SENAO": "SENAO", "FIM": "FIM",
    "E": "E", "OU": "OU", "NAO": "NAO", "HORARIO": "um horário HH:MM",
    "NUMERO": "um número", "TEXTO": "um texto entre aspas", "NOME": "um nome"
}

def erro_amigavel(e, fonte):
    contexto = e.get_context(fonte).rstrip() if hasattr(e, 'get_context') else ""
    if isinstance(e, UnexpectedCharacters):
        return "léxica", f"ERRO LÉXICO na linha {e.line}, coluna {e.column}: caractere inválido '{fonte[e.pos_in_stream]}'.\n\n{contexto}"
    if isinstance(e, UnexpectedToken):
        esp = sorted({NOMES_AMIGAVEIS.get(x, x) for x in e.expected})
        return "sintática", f"ERRO SINTÁTICO na linha {e.line}, coluna {e.column}: encontrei '{e.token}', esperava: {', '.join(esp)}.\n\n{contexto}"
    if isinstance(e, UnexpectedEOF):
        return "sintática", "ERRO SINTÁTICO: fim de arquivo inesperado (faltou FIM?)."
    return "desconhecida", str(e)

@dataclass
class ResultadoCompilacaoPet:
    ok: bool = False
    fase_erro: str = None
    mensagens: list = field(default_factory=list)
    avisos: list = field(default_factory=list)
    ast_otimizada: object = None
    relatorio_otimizacao: list = field(default_factory=list)
    bytecode: list = field(default_factory=list)

def compilar_pet(fonte, otimizar=True):
    res = ResultadoCompilacaoPet()
    try:
        arvore = parser_pet.parse(fonte)
    except (UnexpectedCharacters, UnexpectedToken, UnexpectedEOF) as e:
        res.fase_erro, msg = erro_amigavel(e, fonte)
        res.mensagens.append(msg)
        return res

    ast = ConstrutorASTPet(fonte).transform(arvore)
    erros, res.avisos = AnalisadorSemanticoPet().analisar(ast)
    if erros:
        res.fase_erro, res.mensagens = "semântica", erros
        return res

    otm = OtimizadorPet()
    res.ast_otimizada = otm.otimizar(ast) if otimizar else ast
    res.relatorio_otimizacao = otm.relatorio
    res.bytecode = GeradorDeCodigoPet().gerar(res.ast_otimizada)
    res.ok = True
    return res

print("Pipeline de compilar_pet pronto!")

Pipeline de compilar_pet pronto!


## 11) Bateria de Testes Automáticos (RF6)

In [27]:
# CÉLULA 11: Bateria de Testes do Projeto (RF6)
CASOS_DE_TESTE_PET = [
    ("1. Regra simples com Horário e SENAO",
     'QUANDO relogio >= 08:00 ENTAO dispensar(comedouro, 100) SENAO notificar("Aguardando horário") FIM', "ok"),
    ("2. Controle de Climatizador e Iluminação",
     'QUANDO temperatura > 26 + 2 ENTAO ajustar(climatizador, 22) SENAO desligar(climatizador) FIM', "ok"),
    ("3. Verificação de Status do Pet",
     'QUANDO status_pet == "brincando" ENTAO ligar(camera) FIM', "ok"),
    ("4. Nível de água e troca de água",
     'QUANDO nivel_agua < 20 ENTAO trocar_agua(fonte_agua) FIM', "ok"),
    ("5. Símbolo inválido (@)",
     'QUANDO presenca_pet @ VERDADEIRO ENTAO ligar(camera) FIM', "léxica"),
    ("6. Horário inválido (25:00)",
     'QUANDO relogio == 25:00 ENTAO dispensar(comedouro, 50) FIM', "léxica"),
    ("7. Faltou o fechamento FIM",
     'QUANDO presenca_pet ENTAO ligar(camera)', "sintática"),
    ("8. Bloco SENAO sem nenhuma ação",
     'QUANDO nivel_agua < 10 ENTAO trocar_agua(fonte_agua) SENAO FIM', "sintática"),
    ("9. Comparador malformado",
     'QUANDO temperatura >= > 25 ENTAO ligar(climatizador) FIM', "sintática"),
    ("10. Sensor inexistente",
     'QUANDO nivel_bateria < 10 ENTAO notificar("Bateria baixa") FIM', "semântica"),
    ("11. Dispositivo incompatível dentro do SENAO",
     'QUANDO presenca_pet ENTAO ligar(camera) SENAO dispensar(camera, 100) FIM', "semântica"),
    ("12. Comparação de Tipos Incompatíveis",
     'QUANDO temperatura == "frio" ENTAO ligar(climatizador) FIM', "semântica"),
    ("13. Aviso: Ações repetidas na mesma regra",
     'QUANDO presenca_pet ENTAO ligar(camera); ligar(camera) FIM', "ok"),
    ("14. Aviso: Condição duplicada entre regras",
     'QUANDO nivel_agua < 10 ENTAO trocar_agua(fonte_agua) FIM QUANDO nivel_agua < 10 ENTAO notificar("Água baixa") FIM', "ok")
]

def rodar_bateria():
    linhas, aprovados = [], 0
    for nome, fonte, esperado in CASOS_DE_TESTE_PET:
        res = compilar_pet(fonte)
        obtido = "ok" if res.ok else res.fase_erro
        passou = (obtido == esperado)
        if passou: aprovados += 1
        msg = "-" if res.ok else res.mensagens[0].splitlines()[0][:70]
        linhas.append(("✔" if passou else "✖", nome, esperado, obtido, msg))
    tabela(["", "Caso de Teste", "Esperado", "Obtido", "Primeira mensagem"], linhas,
           titulo=f"Bateria de Testes: {aprovados}/{len(CASOS_DE_TESTE_PET)} aprovados")

rodar_bateria()

,Caso de Teste,Esperado,Obtido,Primeira mensagem
✔,1. Regra simples com Horário e SENAO,ok,ok,-
✔,2. Controle de Climatizador e Iluminação,ok,ok,-
✔,3. Verificação de Status do Pet,ok,ok,-
✔,4. Nível de água e troca de água,ok,ok,-
✔,5. Símbolo inválido (@),léxica,léxica,"ERRO LÉXICO na linha 1, coluna 21: caractere inválido '@'."
✔,6. Horário inválido (25:00),léxica,léxica,"ERRO LÉXICO na linha 1, coluna 21: caractere inválido ':'."
✔,7. Faltou o fechamento FIM,sintática,sintática,"ERRO SINTÁTICO na linha 1, coluna 39: encontrei '', esperava: FIM, SEM"
✔,8. Bloco SENAO sem nenhuma ação,sintática,sintática,"ERRO SINTÁTICO na linha 1, coluna 60: encontrei 'FIM', esperava: um no"
✔,9. Comparador malformado,sintática,sintática,"ERRO SINTÁTICO na linha 1, coluna 23: encontrei '>', esperava: FALSO,"
✔,10. Sensor inexistente,semântica,semântica,"Linha 1, col 8: sensor 'nivel_bateria' não existe. Você quis dizer 'ni"


## 12) Simulação de Um Dia no Pet House (RF6)

In [28]:
# CÉLULA 12: Simulação de Eventos no Ambiente CasaPet (RF6)
PROGRAMA_SIMULACAO_PET = """
# Regra 1: Alimentação no horário programado
QUANDO relogio >= 08:00 E relogio <= 09:00 ENTAO
    dispensar(comedouro, 100);
    notificar("Hora do café do pet!")
SENAO
    notificar("Fora do horário de refeição")
FIM

# Regra 2: Conforto térmico do Pet
QUANDO temperatura > 25 ENTAO
    ajustar(climatizador, 22)
SENAO
    desligar(climatizador)
FIM

# Regra 3: Monitoramento de Presença
QUANDO presenca_pet ENTAO
    ligar(camera)
SENAO
    desligar(camera)
FIM
"""

SIMULACAO_EVENTOS_PET = [
    ("relogio", 420),
    ("relogio", 480),
    ("presenca_pet", True),
    ("temperatura", 28),
    ("temperatura", 23),
    ("relogio", 550),
    ("presenca_pet", False),
    ("status_pet", "comendo"),
    ("nivel_agua", 15),
    ("temperatura", 29),
    ("temperatura", 21)
]

casa_pet = CasaPet()
compilado = compilar_pet(PROGRAMA_SIMULACAO_PET)

if compilado.ok:
    central_pet = CentralAutomaçaoPet(compilado.bytecode, casa_pet)
    linhas_simulacao = []

    for sensor, valor in SIMULACAO_EVENTOS_PET:
        antes_log = len(casa_pet.log)
        disparadas = central_pet.evento(sensor, valor)
        novos_logs = casa_pet.log[antes_log:]

        val_fmt = f"{valor//60:02d}:{valor%60:02d}" if sensor == "relogio" else str(valor)
        linhas_simulacao.append((
            f"{sensor} = {val_fmt}",
            ", ".join(disparadas) if disparadas else "-",
            " | ".join(novos_logs) if novos_logs else "-"
        ))

    tabela(["Evento no Sensor", "Regras e Blocos Disparados", "Ações Executadas no Sistema"],
           linhas_simulacao, titulo="Simulação de um Dia no Pet House (Borda de Subida e Descida)")
else:
    caixa("Falha ao compilar o programa de simulação!", "erro")

Evento no Sensor,Regras e Blocos Disparados,Ações Executadas no Sistema
relogio = 07:00,-,-
relogio = 08:00,#1 (ENTAO),[08:00] DISPENSAR: 100g de ração em 'comedouro' | [08:00] NOTIFICAÇÃO: Hora do café do pet!
presenca_pet = True,#3 (ENTAO),[08:00] LIGAR: 'camera'
temperatura = 28,#2 (ENTAO),[08:00] AJUSTAR: 'climatizador' para 22°C
temperatura = 23,#2 (SENAO),[08:00] DESLIGAR: 'climatizador'
relogio = 09:10,#1 (SENAO),[09:10] NOTIFICAÇÃO: Fora do horário de refeição
presenca_pet = False,#3 (SENAO),[09:10] DESLIGAR: 'camera'
status_pet = comendo,-,-
nivel_agua = 15,-,-
temperatura = 29,#2 (ENTAO),[09:10] AJUSTAR: 'climatizador' para 22°C


In [31]:
# CÉLULA 13: PAINEL COMPLETO (CasaPet Studio 2.0 — Estilo Premium)

def montar_cards_html_pet(casa):
    d = casa.dispositivos

    def badge(status, ativo):
        cor_bg = "#16a34a" if ativo else "#dc2626"
        return f"<span style='background:{cor_bg}; color:#fff; padding:2px 8px; border-radius:12px; font-size:11px; font-weight:600;'>{status}</span>"

    itens = [
        ("🥣", "Comedouro", f"{d['comedouro']['porcoes']}g servidas", "#2563eb"),
        ("🌊", "Fonte de Água", badge("LIGADA" if d['fonte_agua']['ligado'] else "DESLIGADA", d['fonte_agua']['ligado']), ""),
        ("❄️", "Climatizador", badge(f"{d['climatizador']['nivel']}°C" if d['climatizador']['ligado'] else "DESLIGADO", d['climatizador']['ligado']), ""),
        ("📹", "Câmera Pet", badge("LIGADA" if d['camera']['ligado'] else "DESLIGADA", d['camera']['ligado']), ""),
        ("🥛", "Tigela de Água", badge("LIMPA" if d['tigela_agua']['limpa'] else "SUJA", d['tigela_agua']['limpa']), "")
    ]

    cards = ""
    for icone, nome, valor, cor_txt in itens:
        val_html = f"<b style='color:{cor_txt}; font-size:12px;'>{valor}</b>" if cor_txt else valor
        cards += f"""
        <div style='display:inline-block; width:135px; margin:5px; padding:12px 8px; border:1px solid #e2e8f0; border-radius:12px; text-align:center; background:#ffffff; box-shadow:0 4px 6px -1px rgba(0,0,0,0.05); font-family:Segoe UI, sans-serif;'>
            <div style='font-size:30px; margin-bottom:4px;'>{icone}</div>
            <div style='font-weight:700; color:#1e293b; font-size:13px;'>{nome}</div>
            <div style='margin-top:6px;'>{val_html}</div>
        </div>
        """
    return f"<div style='display:flex; flex-wrap:wrap; justify-content:flex-start; margin-top:8px;'>{cards}</div>"

def diario_html_pet(logs):
    if not logs:
        linhas_html = "<tr><td colspan='2' style='padding:8px; color:#94a3b8; text-align:center;'>Nenhuma ação executada ainda.</td></tr>"
    else:
        linhas_html = "".join([
            f"<tr style='border-bottom:1px solid #f1f5f9;'><td style='padding:6px 10px; color:#0284c7; font-weight:600; font-family:Consolas, monospace; width:90px;'>{log.split(']')[0]}]</td><td style='padding:6px 10px; color:#334155; font-family:Consolas, monospace;'>{log.split(']')[1] if ']' in log else log}</td></tr>"
            for log in logs[-8:]
        ])

    return f"""
    <div style='margin-top:12px; font-family:Segoe UI, sans-serif;'>
        <b style='color:#0f172a; font-size:14px;'>📋 Diário de Eventos do Pet House</b>
        <div style='border:1px solid #e2e8f0; border-radius:8px; overflow:hidden; margin-top:6px; background:#fff; box-shadow:0 1px 3px rgba(0,0,0,0.05);'>
            <table style='width:100%; border-collapse:collapse; font-size:12px;'>
                <tr style='background:#111111; color:#ffffff; text-align:left;'>
                    <th style='padding:6px 10px; width:90px;'>Horário</th>
                    <th style='padding:6px 10px;'>Ação do Sistema</th>
                </tr>
                {linhas_html}
            </table>
        </div>
    </div>
    """

# Componentes de Entrada
editor_pet = widgets.Textarea(
    value=PROGRAMA_SIMULACAO_PET.strip(),
    layout=widgets.Layout(width="98%", height="220px", font_family="Consolas")
)
chk_otimizar_pet = widgets.Checkbox(value=True, description="Otimizar Bytecode")
btn_compilar_pet = widgets.Button(description="▶ Compilar Regras", button_style="success", layout=widgets.Layout(width="180px"))
saida_status_pet = widgets.Output()

saidas_fases = [widgets.Output() for _ in range(4)]
abas_pet = widgets.Tab(children=saidas_fases)
for idx, tit in enumerate(["🔎 AST Otimizada", "🛡️ Semântica", "⚡ Otimização", "💾 Bytecode"]):
    abas_pet.set_title(idx, tit)

# Sensores
s_relogio = widgets.IntSlider(value=480, min=0, max=1439, step=15, description="Relógio:", layout=widgets.Layout(width="90%"))
s_temp = widgets.IntSlider(value=22, min=-10, max=50, description="Temperatura:", layout=widgets.Layout(width="90%"))
s_agua = widgets.IntSlider(value=80, min=0, max=100, description="Água (%):", layout=widgets.Layout(width="90%"))
s_racao = widgets.IntSlider(value=500, min=0, max=2000, step=50, description="Ração (g):", layout=widgets.Layout(width="90%"))
s_status = widgets.Dropdown(options=["dormindo", "brincando", "comendo", "ausente"], value="dormindo", description="Status Pet:", layout=widgets.Layout(width="90%"))
s_presenca = widgets.Checkbox(value=False, description="Pet Presente na Área")

painel_pet = widgets.HTML()
diario_pet = widgets.HTML()
estado_pet = {"central": None, "bytecode": None}

def atualizar_interface_pet():
    if estado_pet["central"] is None:
        painel_pet.value = "<div style='color:#ef4444; padding:10px; font-family:Segoe UI;'>⚠️ Compile um programa válido na aba IDE para conectar a casa.</div>"
        diario_pet.value = ""
        return
    casa = estado_pet["central"].casa
    painel_pet.value = montar_cards_html_pet(casa)
    diario_pet.value = diario_html_pet(casa.log)

def ao_compilar_studio(_):
    for s in saidas_fases + [saida_status_pet]:
        s.clear_output()
    res = compilar_pet(editor_pet.value, otimizar=chk_otimizar_pet.value)

    with saida_status_pet:
        if res.ok:
            caixa(f"✔ Compilado com Sucesso! {len(res.bytecode)} regra(s) gerada(s).", "ok")
            estado_pet["bytecode"] = res.bytecode
            casa = CasaPet()
            estado_pet["central"] = CentralAutomaçaoPet(res.bytecode, casa)
            atualizar_interface_pet()
        else:
            caixa(f"✖ Falha na Fase {res.fase_erro.upper()}:\n" + "\n".join(res.mensagens), "erro")
            estado_pet["central"] = None
            atualizar_interface_pet()

    with saidas_fases[0]:
        if res.ast_otimizada: print(repr(res.ast_otimizada))
    with saidas_fases[1]:
        if res.avisos: caixa("\n".join(res.avisos), "aviso")
        else: print("✔ Nenhum aviso semântico.")
    with saidas_fases[2]:
        if res.relatorio_otimizacao: tabela(["Técnica", "Antes", "Depois"], res.relatorio_otimizacao)
        else: print("Nenhuma otimização necessária.")
    with saidas_fases[3]:
        if res.bytecode:
            for r in res.bytecode:
                print(f"--- Regra #{r.numero} ---")
                print("Condição:", r.condicao)
                print("Bloco ENTAO:", r.acoes_entao)
                print("Bloco SENAO:", r.acoes_senao)
                print()

def ao_mudar_sensor_studio(change):
    if estado_pet["central"] is not None:
        c = estado_pet["central"]
        c.casa.sensores.update({
            "relogio": s_relogio.value,
            "temperatura": s_temp.value,
            "nivel_agua": s_agua.value,
            "peso_racao": s_racao.value,
            "status_pet": s_status.value,
            "presenca_pet": s_presenca.value
        })
        c.avaliar_todas()
        atualizar_interface_pet()

for w in [s_relogio, s_temp, s_agua, s_racao, s_status, s_presenca]:
    w.observe(ao_mudar_sensor_studio, names="value")

btn_compilar_pet.on_click(ao_compilar_studio)

# Agrupamento do Layout
box_controles = widgets.VBox([
    widgets.HTML("<b style='font-family:Segoe UI; color:#0f172a;'>🎛️ Painel de Sensores</b>"),
    s_relogio, s_temp, s_agua, s_racao, s_status, s_presenca
], layout=widgets.Layout(width="320px", padding="10px", border="1px solid #e2e8f0", border_radius="10px", background="#f8fafc"))

box_simulacao = widgets.VBox([painel_pet, diario_pet], layout=widgets.Layout(width="100%"))

box_ide = widgets.VBox([
    editor_pet,
    widgets.HBox([btn_compilar_pet, chk_otimizar_pet]),
    saida_status_pet,
    abas_pet
])

box_casa = widgets.HBox([box_controles, box_simulacao], layout=widgets.Layout(gap="15px"))

studio_tabs = widgets.Tab(children=[box_ide, box_casa])
studio_tabs.set_title(0, "💻 IDE Compilador")
studio_tabs.set_title(1, "🐾 CasaPet Simulador")

display(HTML("<h3 style='font-family:Segoe UI; color:#0f172a; margin-bottom:12px;'>🐾 CasaPet Studio 2.0</h3>"))
display(studio_tabs)
ao_compilar_studio(None)